# Meningitis (Kaggle) — Clasificación con faltantes

## En qué consiste
- Pacientes con variables clínicas y laboratorio; hay **valores faltantes**.
- **Objetivo ejemplo**: predecir **`Outcome`** (recuperación, etc.) o **`Diagnosis`**.

## Pasos
1. Quitar **Patient_ID**.
2. Codificar **Outcome** con códigos enteros.
3. Imputación + one-hot + escalado; antes/después; 10 filas.

In [ ]:
import sys
from pathlib import Path

# --- Localizar la raiz del proyecto (carpeta donde estan .env y load_project_env.py) ---
def _project_root() -> Path:
    """Busca hacia arriba desde el directorio de trabajo hasta encontrar load_project_env.py."""
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "load_project_env.py").exists():
            return cand
    return p

ROOT = _project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from load_project_env import load_env, data_path

load_env()
print("PROJECT_ROOT =", ROOT)


In [ ]:
import pandas as pd
import numpy as np

path = data_path("DATA_MENINGITIS")
df_original = pd.read_csv(path)
print("Forma:", df_original.shape)
print("Columnas:", list(df_original.columns))

if "Patient_ID" in df_original.columns:
    df_original = df_original.drop(columns=["Patient_ID"])
    print("[ELIMINADA] Patient_ID — identificador.")

if "Outcome" in df_original.columns:
    y = pd.Categorical(df_original["Outcome"]).codes
    X = df_original.drop(columns=["Outcome"])
    print("[OBJETIVO] Outcome codificado como enteros 0..k-1.")
else:
    X = df_original
    y = np.zeros(len(X))

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
print("Numericas:", num_cols)
print("Categoricas:", cat_cols)

n_preview = min(12, X.shape[1])
print("\n=== MUESTRA: 10 filas de X (primeras", n_preview, "columnas) ===")
print(X.iloc[:10, :n_preview].to_string())


## Pipeline de `sklearn` (qué hace cada pieza)

| Pieza | Rol |
|-------|-----|
| **SimpleImputer (median)** | Sustituye NaN en numéricas por la mediana (robusto a outliers). |
| **StandardScaler** | Normaliza: \((x - \mu) / \sigma\) por columna → media ~0, varianza ~1. |
| **SimpleImputer (most_frequent)** | Sustituye NaN en categóricas por la categoría más frecuente. |
| **OneHotEncoder** | Convierte cada categoría en columnas 0/1 (una por valor visto al entrenar). |

La siguiente celda **define** `build_preprocessor`. La siguiente **ajusta** (`fit_transform`) y muestra **antes/después** en numéricas y una **muestra de la matriz final**.

In [ ]:
from sklearn.compose import ColumnTransformer  # Une transformaciones por tipo de columna
from sklearn.impute import SimpleImputer  # Rellena valores faltantes
from sklearn.pipeline import Pipeline  # Encadena pasos: imputer -> scaler / one-hot
from sklearn.preprocessing import OneHotEncoder, StandardScaler  # Escalado y codificacion categorica


def build_preprocessor(numeric_cols, categorical_cols):
    """Numericos: mediana + StandardScaler. Categoricos: moda + one-hot. Listas vacias = se omite ese bloque."""
    # Sub-pipeline numerico: primero imputar, luego llevar a escala comun
    num_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),  # NaN -> mediana de la columna
            ("scaler", StandardScaler()),  # (x - media) / std por columna
        ]
    )
    # Sub-pipeline categorico: imputar texto y expandir a columnas binarias
    cat_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),  # NaN -> categoria mas frecuente
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ),  # Una columna 0/1 por valor distinto visto al hacer fit
        ]
    )
    transformers = []  # Lista de tuplas (nombre, pipeline, lista de columnas)
    if numeric_cols:  # Si hay al menos una columna numerica
        transformers.append(("num", num_pipe, numeric_cols))
    if categorical_cols:  # Si hay al menos una columna no numerica
        transformers.append(("cat", cat_pipe, categorical_cols))
    if not transformers:
        raise ValueError("No hay columnas numericas ni categoricas para transformar.")
    # remainder='drop' ignora columnas no listadas (aqui no deberia haber)
    return ColumnTransformer(transformers=transformers, remainder="drop")


In [ ]:

# ========== PASO FINAL: entrenar transformaciones y obtener matriz para ML ==========
# fit_transform: aprende medianas/modas sobre X y devuelve la matriz ya transformada
prep = build_preprocessor(num_cols, cat_cols)  # Construye el objeto (aun no aprende estadisticos)
X_matrix = prep.fit_transform(X)  # Aqui se aprende y se aplica: salida 2D numpy

# Nombres de columnas de salida (prefijos num__ y cat__)
feature_names = list(prep.get_feature_names_out())
print("Shape X_matrix (listo para red neuronal / sklearn):", X_matrix.shape)
print("Total de features despues de one-hot:", len(feature_names))

pd.set_option("display.max_columns", 14)  # Limitar columnas al imprimir tablas
pd.set_option("display.width", 220)

# --- Comparar ANTES y DESPUES solo en columnas NUMERICAS (efecto de StandardScaler) ---
num_pipe_fitted = prep.named_transformers_.get("num")  # None si no hubo columnas numericas
if num_cols and num_pipe_fitted is not None:
    X_num_scaled = num_pipe_fitted.transform(X[num_cols])  # Mismas filas, solo bloque numerico escalado
    antes_num = X[num_cols].iloc[:10].copy()  # 10 filas tal cual en el CSV
    despues_num = pd.DataFrame(
        X_num_scaled[:10],
        columns=num_cols,
        index=antes_num.index,
    )
    print("\n=== NORMALIZACION (StandardScaler sobre numericas): ANTES — 10 filas, valores ORIGINALES ===")
    print(antes_num.to_string())
    print("\n=== NORMALIZACION: DESPUES — mismas 10 filas, ESCALADAS (media global ~0, std ~1 por columna) ===")
    print(despues_num.to_string())
    print("\n=== Resumen estadistico numericas ANTES (mean, std, min, max) ===")
    print(X[num_cols].describe().T[["mean", "std", "min", "max"]].head(20).to_string())
    print("\n=== Resumen estadistico numericas DESPUES del escalado (deberian tener mean~0, std~1) ===")
    print(pd.DataFrame(X_num_scaled, columns=num_cols).describe().T[["mean", "std", "min", "max"]].head(20).to_string())
else:
    print("\n(No hay columnas numericas en X o solo hay categoricas; no se aplica StandardScaler en bloque num.)")

# --- Salida mezclada: primeras columnas de la matriz final (numeros escalados + one-hot) ---
n_show = min(20, X_matrix.shape[1])
df_final_muestra = pd.DataFrame(
    X_matrix[:10, :n_show],
    columns=[str(feature_names[i])[:45] for i in range(n_show)],
)
print("\n=== MATRIZ FINAL: 10 filas x primeras " + str(n_show) + " columnas (nombres truncados; incluye one-hot) ===")
print(df_final_muestra.to_string())

# Etiqueta objetivo (regresion o clasificacion)
if y is None:
    print("\n[y] No hay vector objetivo en este cuaderno (solo features).")
else:
    print("\n[y] Forma del vector objetivo:", getattr(y, "shape", type(y)))
    print("[y] Primeras 10 etiquetas:", y[:10] if hasattr(y, "__getitem__") else y)


## Cómo leer la salida del último bloque de código

| Sección | Qué verás |
|---------|-----------|
| **MUESTRA 10 filas de X** | Tus **features crudas** (antes de imputar/escalar/one-hot). Solo se muestran las primeras columnas para que quepa en pantalla. |
| **ANTES (normalización)** | Valores **originales** de las columnas **numéricas** (10 filas). |
| **DESPUÉS (normalización)** | Mismas columnas tras **StandardScaler**: cada columna tiene media ≈ 0 y desviación ≈ 1 **sobre todo el conjunto** (útil para redes y SVM). |
| **Resumen estadístico ANTES/DESPUÉS** | `mean`, `std`, `min`, `max` para comprobar el efecto del escalado. |
| **MATRIZ FINAL (10 × N)** | Matriz que **entra al modelo**: bloque numérico ya escalado + columnas **one-hot** de categorías (nombres tipo `cat__Sex_Male`). |
| **y** | Etiquetas: precio, clase, etc., según el dataset. |

**Nota:** Las columnas categóricas no aparecen en la tabla “ANTES/DESPUÉS” numérica; su información pasa a columnas 0/1 en la **MATRIZ FINAL**.